# Demo 03 : Code review

We send a small patch to independent reviewers, then ask a judge to resolve their findings. A deterministic checker enforces required evidence in the final review. Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents==0.7.24", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · code-review

Parallel reviewers receive different questions about one patch. A judge sees both findings and the patch before selecting actionable issues.

In [2]:
import operator
from typing import Annotated

from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel
from typing_extensions import TypedDict

PATCH = '''
def reserve_part(db, part_no, quantity):
    row = db.execute("SELECT stock FROM parts WHERE part_no = ?", (part_no,)).fetchone()
    if row is None:
        return True
    db.execute("UPDATE parts SET stock = stock - ? WHERE part_no = ?", (quantity, part_no))
    return True
'''

class ReviewState(TypedDict):
    patch: str
    findings: Annotated[list[dict], operator.add]
    verdict: str

def review(role):
    def run(state):
        text = llm.invoke([
            SystemMessage("Review only the assigned concern. Report concrete issues with a short quote from the patch. If none, say none."),
            HumanMessage(f"Concern: {role}\nPatch:\n{state['patch']}")
        ]).content
        return {"findings": [{"role": role, "text": text}]}
    return run

def dispatch(state):
    return [Send(role, {"patch": state["patch"], "findings": [], "verdict": ""})
            for role in ["correctness", "database_safety"]]

print(PATCH)


def reserve_part(db, part_no, quantity):
    row = db.execute("SELECT stock FROM parts WHERE part_no = ?", (part_no,)).fetchone()
    if row is None:
        return True
    db.execute("UPDATE parts SET stock = stock - ? WHERE part_no = ?", (quantity, part_no))
    return True



In [3]:
class Judge(BaseModel):
    accepted_findings: list[str]
    rationale: str

def judge(state):
    findings = llm.with_structured_output(Judge).invoke(
        "Keep only findings supported by the patch. Include quoted evidence for each accepted issue. "
        f"Patch:\n{state['patch']}\nReviews:\n{state['findings']}")
    return {"verdict": "\n".join(findings.accepted_findings) + "\n" + findings.rationale}

graph = StateGraph(ReviewState)
graph.add_node("correctness", review("correctness and edge cases"))
graph.add_node("database_safety", review("database safety and transaction behavior"))
graph.add_node("judge", judge)
graph.add_conditional_edges(START, dispatch, ["correctness", "database_safety"])
graph.add_edge(["correctness", "database_safety"], "judge")
graph.add_edge("judge", END)
review_result = graph.compile().invoke({"patch": PATCH, "findings": [], "verdict": ""})
print("reviewers:", [item["role"] for item in review_result["findings"]])
print("judge:\n", review_result["verdict"])

reviewers: ['correctness and edge cases', 'database safety and transaction behavior']
judge:
 A missing part is reported as successfully reserved: `if row is None: return True`.
The function subtracts the requested quantity without checking availability, so a request exceeding stock can make inventory negative: `UPDATE parts SET stock = stock - ? WHERE part_no = ?`.
Negative quantities increase inventory because the function subtracts the supplied quantity without validating it: `UPDATE parts SET stock = stock - ? WHERE part_no = ?`. 
The patch directly supports the missing-part success path and the unchecked stock arithmetic. The claimed concurrency race is not accepted: the patch alone does not establish the database's transaction or isolation behavior.


A code checker can reject structural mistakes without another model call. `no-mistakes` applies this idea to a coding fleet as a pipeline of review, tests, documentation, lint, push, pull request and CI checks.

In [4]:
REQUIRED = ["reserve_part", "SELECT", "UPDATE"]
checks = {token: token in PATCH for token in REQUIRED}
print("pipeline checks:", checks)
print("checker:", "pass" if all(checks.values()) else "reject")

pipeline checks: {'reserve_part': True, 'SELECT': True, 'UPDATE': True}
checker: pass
